## Task 1c: Hyperparameter selection with stratified k-fold CV
We use stratified 5-fold CV on the training data to select the kernel, C, γ and class weights for both OvO and OvA SVMs. Preprocessing is fit inside each training fold only. The same folds are stored to reuse in Task 3.

**Search grid**
- Kernel = linear: C ∈ {0.1, 1, 10}
- Kernel = rbf: C ∈ {0.1, 1, 10, 100}, γ ∈ {0.01, 0.1, 1, 'scale'}
- Class weight ∈ {None, 'balanced'}

Selection metric: mean CV macro-F1 (accuracy is reported alongside, because the classes are imbalanced).

In [7]:
from itertools import combinations, product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

train = load_data(TRAIN_PATH)
FEATS = NUM_COLS + ORD_COLS + NOM_COLS
y_all = train[TARGET].values
classes = np.array(sorted(np.unique(y_all)))
K = len(classes)

# Same folds are reused for every configuration (and later in Task 3)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(train[FEATS], y_all))

# Preprocessing fit on each training fold only, shared across configurations
fold_data = []
for tr_idx, va_idx in folds:
    pre = build_preprocessor()
    X_tr = pre.fit_transform(train.iloc[tr_idx][FEATS])
    X_va = pre.transform(train.iloc[va_idx][FEATS])
    fold_data.append((X_tr, y_all[tr_idx], X_va, y_all[va_idx]))

C:\Users\Asus\AppData\Local\Temp\ipykernel_17160\2255881943.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=["object"]).columns:


In [8]:
def ovo_fit_predict(X_tr, y_tr, X_te, **svm_params):
    cls = np.array(sorted(np.unique(y_tr)))
    k = len(cls)
    prior = np.array([(y_tr == c).mean() for c in cls])
    votes = np.zeros((X_te.shape[0], k), dtype=int)
    scores = np.zeros((X_te.shape[0], k))
    for i, j in combinations(range(k), 2):
        m = (y_tr == cls[i]) | (y_tr == cls[j])
        clf = SVC(**svm_params).fit(X_tr[m], (y_tr[m] == cls[j]).astype(int))
        d = clf.decision_function(X_te)
        votes[d > 0, j] += 1
        votes[d <= 0, i] += 1
        scores[:, j] += d
        scores[:, i] -= d
    pred = np.empty(X_te.shape[0], dtype=int)
    for n in range(X_te.shape[0]):
        cand = np.where(votes[n] == votes[n].max())[0]
        if len(cand) > 1:
            best = cand[np.isclose(scores[n, cand], scores[n, cand].max())]
            if len(best) > 1:
                best = best[prior[best] == prior[best].max()]
            cand = best
        pred[n] = cand[0]
    return cls[pred]


def ova_fit_predict(X_tr, y_tr, X_te, **svm_params):
    cls = np.array(sorted(np.unique(y_tr)))
    scores = np.zeros((X_te.shape[0], len(cls)))
    for k, c in enumerate(cls):
        clf = SVC(**svm_params).fit(X_tr, (y_tr == c).astype(int))
        scores[:, k] = clf.decision_function(X_te)
    return cls[np.argmax(scores, axis=1)]

In [9]:
# Search grid
grid = []
for cw in [None, "balanced"]:
    for c in [0.1, 1, 10]:
        grid.append(dict(kernel="linear", C=c, gamma="-", class_weight=cw))
    for c, g in product([0.1, 1, 10, 100], [0.01, 0.1, 1, "scale"]):
        grid.append(dict(kernel="rbf", C=c, gamma=g, class_weight=cw))
print("Number of configurations:", len(grid))

rows = []
for method, fp in [("OvO", ovo_fit_predict), ("OvA", ova_fit_predict)]:
    for cfg in grid:
        params = dict(kernel=cfg["kernel"], C=cfg["C"], class_weight=cfg["class_weight"])
        if cfg["kernel"] == "rbf":
            params["gamma"] = cfg["gamma"]
        accs, f1s = [], []
        for X_tr, y_tr, X_va, y_va in fold_data:
            p = fp(X_tr, y_tr, X_va, **params)
            accs.append(accuracy_score(y_va, p))
            f1s.append(f1_score(y_va, p, average="macro"))
        rows.append(dict(method=method, kernel=cfg["kernel"], C=cfg["C"],
                         gamma=cfg["gamma"], class_weight=str(cfg["class_weight"]),
                         acc_mean=np.mean(accs), acc_std=np.std(accs),
                         f1_mean=np.mean(f1s), f1_std=np.std(f1s)))

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUTPUT_DIR, "cv_results.csv"), index=False)

Number of configurations: 38


In [10]:
pd.set_option("display.max_rows", None)
for method in ["OvO", "OvA"]:
    print(f"\n===== {method}: all configurations (sorted by mean macro-F1) =====")
    display(results[results.method == method]
            .sort_values("f1_mean", ascending=False)
            .drop(columns="method").round(4).reset_index(drop=True))

best = results.loc[results.groupby("method")["f1_mean"].idxmax()]
print("\nBest configuration per method:")
display(best.round(4))


===== OvO: all configurations (sorted by mean macro-F1) =====


,kernel,C,gamma,class_weight,acc_mean,acc_std,f1_mean,f1_std
0,rbf,10.0,scale,balanced,0.3303,0.0111,0.3232,0.0130
1,rbf,100.0,0.1,balanced,0.3254,0.0196,0.3193,0.0186
2,rbf,10.0,0.1,balanced,0.3232,0.0129,0.3174,0.0131
3,rbf,100.0,scale,balanced,0.3194,0.0128,0.3129,0.0123
4,rbf,10.0,scale,None,0.3286,0.0134,0.3050,0.0118
5,rbf,1.0,1,balanced,0.3205,0.0157,0.3035,0.0158
6,rbf,1.0,scale,balanced,0.3090,0.0163,0.3024,0.0191
7,rbf,1.0,0.1,balanced,0.3090,0.0161,0.3024,0.0181
8,rbf,100.0,0.01,balanced,0.3063,0.0214,0.3023,0.0211
9,rbf,10.0,0.1,None,0.3254,0.0241,0.3009,0.0242



===== OvA: all configurations (sorted by mean macro-F1) =====


,kernel,C,gamma,class_weight,acc_mean,acc_std,f1_mean,f1_std
0,rbf,1.0,0.1,balanced,0.3194,0.0152,0.3153,0.0155
1,rbf,1.0,scale,balanced,0.3188,0.0221,0.3139,0.0226
2,rbf,10.0,0.1,balanced,0.3199,0.0151,0.3132,0.0155
3,rbf,10.0,scale,balanced,0.3199,0.0115,0.3124,0.0131
4,rbf,100.0,0.1,balanced,0.3161,0.0109,0.3100,0.0092
5,rbf,1.0,0.1,None,0.3406,0.0183,0.3097,0.0176
6,rbf,100.0,0.01,balanced,0.3123,0.0230,0.3089,0.0228
7,rbf,0.1,scale,balanced,0.3090,0.0086,0.3066,0.0101
8,rbf,0.1,0.1,None,0.3368,0.0189,0.3058,0.0184
9,rbf,1.0,scale,None,0.3330,0.0213,0.3034,0.0208



Best configuration per method:


,method,kernel,C,gamma,class_weight,acc_mean,acc_std,f1_mean,f1_std
65,OvA,rbf,1.0,0.1,balanced,0.3194,0.0152,0.3153,0.0155
33,OvO,rbf,10.0,scale,balanced,0.3303,0.0111,0.3232,0.0130


### Task 1c: Results and selected configuration

Each of the 38 configurations was evaluated with stratified 5-fold CV (shuffled, seed 42) for both OvO and OvA. The tables above report the mean and std of accuracy and macro-F1 across folds. Selection was by mean macro-F1.

| Method | Kernel | C | γ | Class weight | Accuracy (mean ± std) | Macro-F1 (mean ± std) |
|---|---|---|---|---|---|---|
| OvO | rbf | 10 | scale | balanced | 0.3303 ± 0.0111 | 0.3232 ± 0.0130 |
| OvA | rbf | 1 | 0.1 | balanced | 0.3194 ± 0.0152 | 0.3153 ± 0.0155 |

**Observations**
- **Class weights:** `balanced` weights dominate the top of both rankings. Without weights, some configurations reach a higher accuracy (e.g. OvO, rbf, C=1, γ=0.01: 0.351) but a much lower macro-F1 (0.215), because they favour the larger classes.
- **Kernel:** rbf beats linear. The best linear model reaches a macro-F1 of 0.278 (OvO) and 0.296 (OvA), against 0.323 and 0.315 for rbf. The decision boundaries are not linear.
- **Underfitting:** very small C with very small γ underfits. OvO with C=0.1, γ=0.01 and no weights has accuracy 0.316 and macro-F1 0.122, which looks like predicting a single class.
- **Overfitting:** C=100 or γ=1 gives somewhat lower scores than the best configurations, but the drop is small.
- **Overall signal:** all accuracies are around 0.30–0.35, so the features carry only weak signal about the segment.
- **Stability:** the gap between the top few configurations (~0.01) is about the size of the fold std (~0.013), so those configurations are not clearly distinguishable. The selected configurations are reasonable choices, not uniquely best ones.